# Airplane Turkish-English Translation Evaluation

This notebook evaluates the MIS 48B airplane-domain translation project. It separates two kinds of evidence:

1. **Direct in-domain comparison**: every selected model is run on the same held-out airplane translation examples from this project. This is the main scientific comparison.
2. **External published benchmarks**: public BLEU/chrF values from model cards and WMT papers. These are useful context, but they are not directly comparable unless the dataset and metric setup match.

Default behavior evaluates a stratified 300-example sample to keep Colab runtime manageable. Set `RUN_FULL_TEST_SET = True` to use the entire held-out split.

In [ ]:
# Runtime setup
# In Colab, run this cell first. Restart the runtime if Colab asks after installation.

!pip install -q -U \
    "transformers>=4.45" \
    "accelerate>=0.33" \
    "datasets>=2.20" \
    "peft>=0.12" \
    "bitsandbytes>=0.43" \
    "sacrebleu>=2.4" \
    "bert-score>=0.3.13" \
    "pandas>=2.0" \
    "numpy>=1.24" \
    "matplotlib>=3.7" \
    "seaborn>=0.13" \
    "tqdm>=4.66" \
    "huggingface_hub>=0.24" \
    "tabulate>=0.9"

In [ ]:
# Imports and global configuration

from __future__ import annotations

import gc
import json
import math
import os
import re
import time
import unicodedata
from datetime import date
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import load_dataset
from huggingface_hub import login
from sacrebleu.metrics import BLEU, CHRF, TER
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

SEED = 42
RUN_FULL_TEST_SET = False
EVAL_SAMPLE_SIZE = 300
RUN_SMOKE_TEST = True
FORCE_REGENERATE_PREDICTIONS = False

BASE_LLAMA_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
OPUS_EN_TR_MODEL_ID = "Helsinki-NLP/opus-mt-tc-big-en-tr"
OPUS_TR_EN_MODEL_ID = "Helsinki-NLP/opus-mt-tc-big-tr-en"
NLLB_MODEL_ID = "facebook/nllb-200-distilled-600M"

LOAD_IN_4BIT = torch.cuda.is_available()
MAX_NEW_TOKENS = 80
GENERATION_BATCH_SIZE_SEQ2SEQ = 16
BERTSCORE_BATCH_SIZE = 16

TRANSLATION_SYSTEM_PROMPT = """You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations.

Your task is to translate the user's sentence between Turkish and English.

Rules:
- If the input is Turkish, translate it into natural English.
- If the input is English, translate it into natural Turkish.
- Preserve the meaning, politeness level, urgency, and speaker intent.
- Use simple, clear, practical language suitable for airplane passengers and cabin crew.
- Do not add explanations.
- Do not answer the user's request.
- Do not roleplay.
- Only return the translated sentence.
- For emergency sentences, keep the translation direct and accurate.
- For polite requests, preserve politeness naturally.
- For announcements or crew instructions, use clear formal language.
"""

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Project paths
# Works in Colab when the project is in Google Drive, or when the folder is uploaded to /content.

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped:", exc)

CANDIDATE_PROJECT_ROOTS = [
    Path("/content/drive/MyDrive/MIS48B+"),
    Path("/content/MIS48B+"),
    Path.cwd(),
]

PROJECT_ROOT = next((p for p in CANDIDATE_PROJECT_ROOTS if (p / "airplane_translation_dataset").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find MIS48B+ project root. Upload the MIS48B+ folder to /content "
        "or place it in /content/drive/MyDrive/MIS48B+."
    )

DATA_FILE = PROJECT_ROOT / "airplane_translation_dataset" / "final" / "fine_tune_chat.jsonl"
MERGED_MODEL_DIR = PROJECT_ROOT / "airplane_translation_model" / "merged_final_model"
OUTPUT_DIR = PROJECT_ROOT / "evaluation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.csv"
METRICS_SUMMARY_PATH = OUTPUT_DIR / "metrics_summary.csv"
GROUPED_METRICS_PATH = OUTPUT_DIR / "grouped_metrics.csv"
INTERNET_BENCHMARKS_PATH = OUTPUT_DIR / "internet_benchmarks.csv"
REPORT_PATH = OUTPUT_DIR / "evaluation_report.md"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_FILE:", DATA_FILE)
print("MERGED_MODEL_DIR:", MERGED_MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing dataset file: {DATA_FILE}")
if not MERGED_MODEL_DIR.exists():
    raise FileNotFoundError(f"Missing merged model directory: {MERGED_MODEL_DIR}")

In [ ]:
# Hugging Face authentication
# Llama models can require accepting the model terms on Hugging Face and providing a token.

def get_hf_token() -> str | None:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        for secret_name in ["HF_TOKEN", "HuggingFac-Write", "HUGGINGFACE_TOKEN"]:
            token = userdata.get(secret_name)
            if token:
                os.environ["HF_TOKEN"] = token
                return token
    except Exception:
        pass
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face token found and login attempted.")
else:
    print(
        "No Hugging Face token found. Public models may still load, but base Llama may fail. "
        "If it fails, add a Colab secret named HF_TOKEN or HuggingFac-Write."
    )

In [ ]:
# Load and recreate the held-out split

raw_dataset = load_dataset("json", data_files=str(DATA_FILE), split="train")
split_dataset = raw_dataset.train_test_split(test_size=0.02, seed=SEED)
heldout_dataset = split_dataset["test"]
print(raw_dataset)
print("Held-out examples:", len(heldout_dataset))

USER_PATTERN = re.compile(r"^Translate to (Turkish|English):\s*(.*)$", re.IGNORECASE | re.DOTALL)

def parse_chat_record(record: dict[str, Any], row_id: int) -> dict[str, Any]:
    messages = record["messages"]
    user_text = messages[1]["content"].strip()
    assistant_text = messages[2]["content"].strip()
    match = USER_PATTERN.match(user_text)
    if not match:
        raise ValueError(f"Unexpected user prompt at row {row_id}: {user_text[:120]}")

    target_language = match.group(1).title()
    source_text = match.group(2).strip()
    source_language = "Turkish" if target_language == "English" else "English"
    metadata = record.get("metadata") or {}

    return {
        "row_id": row_id,
        "source_text": source_text,
        "reference_text": assistant_text,
        "source_language": source_language,
        "target_language": target_language,
        "direction": f"{source_language}->{target_language}",
        "domain": metadata.get("domain", "unknown"),
        "scenario_group": metadata.get("scenario_group", "unknown"),
        "difficulty": metadata.get("difficulty", "unknown"),
        "tone": metadata.get("tone", "unknown"),
        "speaker": metadata.get("speaker", "unknown"),
        "listener": metadata.get("listener", "unknown"),
        "user_prompt": user_text,
    }

heldout_records = [parse_chat_record(record, idx) for idx, record in enumerate(heldout_dataset)]
heldout_df = pd.DataFrame(heldout_records)

mojibake_pattern = re.compile(r"[ÃÄÅ]")
mojibake_hits = heldout_df[
    heldout_df["source_text"].str.contains(mojibake_pattern, regex=True, na=False)
    | heldout_df["reference_text"].str.contains(mojibake_pattern, regex=True, na=False)
]
if len(mojibake_hits):
    raise ValueError(f"Possible UTF-8 mojibake detected in {len(mojibake_hits)} held-out rows.")

print(heldout_df.head())
print(heldout_df[["direction", "domain", "difficulty", "tone"]].describe(include="all"))

In [ ]:
# Build a fixed stratified evaluation sample

def stratified_sample(df: pd.DataFrame, n: int, strata_cols: list[str], seed: int = SEED) -> pd.DataFrame:
    if n >= len(df):
        return df.copy().reset_index(drop=True)

    work = df.copy()
    work["_stratum"] = work[strata_cols].astype(str).agg(" | ".join, axis=1)
    counts = work["_stratum"].value_counts().sort_index()
    raw_alloc = counts / counts.sum() * n
    alloc = np.floor(raw_alloc).astype(int)
    alloc = alloc.clip(lower=1, upper=counts)

    while int(alloc.sum()) > n:
        reducible = alloc[alloc > 1]
        if reducible.empty:
            break
        idx = (raw_alloc.loc[reducible.index] - alloc.loc[reducible.index]).sort_values().index[0]
        alloc.loc[idx] -= 1

    while int(alloc.sum()) < n:
        expandable = alloc[alloc < counts]
        if expandable.empty:
            break
        idx = (raw_alloc.loc[expandable.index] - alloc.loc[expandable.index]).sort_values(ascending=False).index[0]
        alloc.loc[idx] += 1

    sampled_parts = []
    for stratum, k in alloc.items():
        group = work[work["_stratum"] == stratum]
        sampled_parts.append(group.sample(n=int(k), random_state=seed))

    sampled = pd.concat(sampled_parts, ignore_index=True)
    sampled = sampled.sample(frac=1.0, random_state=seed).drop(columns=["_stratum"]).reset_index(drop=True)
    return sampled

if RUN_FULL_TEST_SET:
    eval_df = heldout_df.copy().reset_index(drop=True)
else:
    eval_df = stratified_sample(heldout_df, EVAL_SAMPLE_SIZE, ["direction", "domain"], SEED)

print("Evaluation examples:", len(eval_df))
display(eval_df.groupby(["direction", "domain"]).size().reset_index(name="n").head(30))

In [ ]:
# Translation helpers

def cleanup_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def model_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

def quantization_config():
    if not LOAD_IN_4BIT or not torch.cuda.is_available():
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_dtype(),
        bnb_4bit_use_double_quant=True,
    )

def build_chat_prompt(tokenizer, source_text: str, target_language: str) -> str:
    messages = [
        {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
        {"role": "user", "content": f"Translate to {target_language}: {source_text}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def translate_causal_batch(model, tokenizer, batch: pd.DataFrame) -> list[str]:
    predictions = []
    for row in batch.itertuples(index=False):
        prompt = build_chat_prompt(tokenizer, row.source_text, row.target_language)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = outputs[0][inputs["input_ids"].shape[-1]:]
        predictions.append(tokenizer.decode(generated, skip_special_tokens=True).strip())
    return predictions

def translate_opus_batch(model, tokenizer, texts: list[str], device: torch.device) -> list[str]:
    outputs = []
    for start in range(0, len(texts), GENERATION_BATCH_SIZE_SEQ2SEQ):
        chunk = texts[start:start + GENERATION_BATCH_SIZE_SEQ2SEQ]
        inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=4)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return [text.strip() for text in outputs]

def translate_nllb_batch(model, tokenizer, batch: pd.DataFrame, device: torch.device) -> list[str]:
    outputs = []
    lang_codes = {"English": "eng_Latn", "Turkish": "tur_Latn"}
    for _, direction_batch in batch.groupby(["source_language", "target_language"], sort=False):
        source_language = direction_batch.iloc[0]["source_language"]
        target_language = direction_batch.iloc[0]["target_language"]
        tokenizer.src_lang = lang_codes[source_language]
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(lang_codes[target_language])
        if forced_bos_token_id is None or forced_bos_token_id == tokenizer.unk_token_id:
            forced_bos_token_id = tokenizer.lang_code_to_id[lang_codes[target_language]]
        texts = direction_batch["source_text"].tolist()
        direction_outputs = []
        for start in range(0, len(texts), GENERATION_BATCH_SIZE_SEQ2SEQ):
            chunk = texts[start:start + GENERATION_BATCH_SIZE_SEQ2SEQ]
            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            with torch.no_grad():
                generated = model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_token_id,
                    max_new_tokens=MAX_NEW_TOKENS,
                    num_beams=4,
                )
            direction_outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        outputs.extend(zip(direction_batch.index.tolist(), [text.strip() for text in direction_outputs]))
    return [prediction for _, prediction in sorted(outputs, key=lambda item: item[0])]

In [ ]:
# Model runners

MODEL_SPECS = [
    {"model_name": "Fine-tuned Llama 3.2 1B", "model_key": "fine_tuned_llama", "kind": "causal", "path": str(MERGED_MODEL_DIR)},
    {"model_name": "Base Llama 3.2 1B", "model_key": "base_llama", "kind": "causal", "path": BASE_LLAMA_MODEL_ID},
    {"model_name": "OPUS-MT tc-big", "model_key": "opus_mt_tc_big", "kind": "opus_pair"},
    {"model_name": "NLLB-200 distilled 600M", "model_key": "nllb_200_distilled_600m", "kind": "nllb", "path": NLLB_MODEL_ID},
]

if any(spec.get("path") == BASE_LLAMA_MODEL_ID for spec in MODEL_SPECS) and not HF_TOKEN:
    raise RuntimeError(
        "Base Llama is enabled but no Hugging Face token was found. Add a Colab secret named "
        "HF_TOKEN or HuggingFac-Write after accepting the model terms on Hugging Face, then rerun."
    )

def run_model_on_dataframe(spec: dict[str, Any], df: pd.DataFrame) -> pd.DataFrame:
    model_key = spec["model_key"]
    model_name = spec["model_name"]
    kind = spec["kind"]
    print(f"\n=== Running {model_name} on {len(df)} examples ===")
    start_time = time.perf_counter()

    if kind == "causal":
        if spec["path"] == BASE_LLAMA_MODEL_ID and not HF_TOKEN:
            raise RuntimeError(
                "Base Llama evaluation needs Hugging Face access. Add a Colab secret named HF_TOKEN "
                "or HuggingFac-Write after accepting the model terms on Hugging Face."
            )
        tokenizer = AutoTokenizer.from_pretrained(spec["path"], use_fast=True, trust_remote_code=True, token=HF_TOKEN)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            spec["path"],
            device_map="auto",
            torch_dtype=model_dtype(),
            quantization_config=quantization_config(),
            trust_remote_code=True,
            token=HF_TOKEN,
        )
        predictions = translate_causal_batch(model, tokenizer, df)
        del model, tokenizer
        cleanup_memory()

    elif kind == "opus_pair":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        predictions_by_index = []
        direction_specs = [
            ("English", "Turkish", OPUS_EN_TR_MODEL_ID),
            ("Turkish", "English", OPUS_TR_EN_MODEL_ID),
        ]
        for source_language, target_language, model_id in direction_specs:
            direction_df = df[(df["source_language"] == source_language) & (df["target_language"] == target_language)]
            if direction_df.empty:
                continue
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
            model.eval()
            direction_predictions = translate_opus_batch(model, tokenizer, direction_df["source_text"].tolist(), device)
            predictions_by_index.extend(zip(direction_df.index.tolist(), direction_predictions))
            del model, tokenizer
            cleanup_memory()
        predictions = [prediction for _, prediction in sorted(predictions_by_index, key=lambda item: item[0])]

    elif kind == "nllb":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(spec["path"], token=HF_TOKEN)
        model = AutoModelForSeq2SeqLM.from_pretrained(
            spec["path"],
            torch_dtype=model_dtype() if torch.cuda.is_available() else torch.float32,
            token=HF_TOKEN,
        ).to(device)
        model.eval()
        predictions = translate_nllb_batch(model, tokenizer, df, device)
        del model, tokenizer
        cleanup_memory()

    else:
        raise ValueError(f"Unknown model kind: {kind}")

    elapsed = time.perf_counter() - start_time
    if len(predictions) != len(df):
        raise ValueError(f"{model_name} produced {len(predictions)} predictions for {len(df)} examples")

    result = df.copy()
    result["model_key"] = model_key
    result["model_name"] = model_name
    result["prediction_text"] = predictions
    result["total_model_seconds"] = elapsed
    result["latency_seconds"] = elapsed / max(len(df), 1)
    return result

def run_all_models(df: pd.DataFrame) -> pd.DataFrame:
    all_predictions = []
    smoke_df = df.head(5).copy()
    if RUN_SMOKE_TEST:
        print("Running 5-example smoke test first.")
        for spec in MODEL_SPECS:
            smoke_predictions = run_model_on_dataframe(spec, smoke_df)
            display(smoke_predictions[["model_name", "direction", "source_text", "reference_text", "prediction_text"]])
    for spec in MODEL_SPECS:
        all_predictions.append(run_model_on_dataframe(spec, df))
    predictions_df = pd.concat(all_predictions, ignore_index=True)
    expected = len(df) * len(MODEL_SPECS)
    if len(predictions_df) != expected:
        raise AssertionError(f"Expected {expected} prediction rows, got {len(predictions_df)}")
    return predictions_df

if PREDICTIONS_PATH.exists() and not FORCE_REGENERATE_PREDICTIONS:
    predictions_df = pd.read_csv(PREDICTIONS_PATH)
    print("Loaded cached predictions:", PREDICTIONS_PATH)
else:
    predictions_df = run_all_models(eval_df)
    predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
    print("Saved predictions to", PREDICTIONS_PATH)

display(predictions_df.head())

In [ ]:
# Metric helper functions

PUNCT_PATTERN = re.compile(r"[^\w\sçğıöşüÇĞİÖŞÜ]", flags=re.UNICODE)
PROMPT_LEAK_PATTERN = re.compile(r"translate to|turkish:|english:|system|assistant|user", re.IGNORECASE)
EXPLANATION_PATTERN = re.compile(r"\b(here is|translation|translated|means|çeviri|tercüme)\b", re.IGNORECASE)

def normalize_for_match(text: Any) -> str:
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", text).strip().casefold()
    text = PUNCT_PATTERN.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def add_behavior_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["prediction_text"] = out["prediction_text"].fillna("").astype(str)
    out["reference_text"] = out["reference_text"].fillna("").astype(str)
    out["source_text"] = out["source_text"].fillna("").astype(str)
    out["normalized_prediction"] = out["prediction_text"].map(normalize_for_match)
    out["normalized_reference"] = out["reference_text"].map(normalize_for_match)
    out["normalized_source"] = out["source_text"].map(normalize_for_match)
    out["exact_match"] = out["prediction_text"].str.strip() == out["reference_text"].str.strip()
    out["normalized_exact_match"] = out["normalized_prediction"] == out["normalized_reference"]
    out["empty_output"] = out["prediction_text"].str.strip().eq("")
    out["source_copy"] = out["normalized_prediction"] == out["normalized_source"]
    out["prompt_leakage"] = out["prediction_text"].str.contains(PROMPT_LEAK_PATTERN, regex=True, na=False)
    out["extra_explanation"] = (
        out["prediction_text"].str.contains(EXPLANATION_PATTERN, regex=True, na=False)
        | out["prediction_text"].str.contains("\n", regex=True, na=False)
    )
    return out

scored_predictions_df = add_behavior_columns(predictions_df)
scored_predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF(word_order=2)
ter_metric = TER()

def corpus_mt_metrics(group: pd.DataFrame) -> dict[str, float]:
    hypotheses = group["prediction_text"].fillna("").astype(str).tolist()
    references = group["reference_text"].fillna("").astype(str).tolist()
    if not hypotheses:
        return {"bleu": np.nan, "chrfpp": np.nan, "ter": np.nan}
    return {
        "bleu": bleu_metric.corpus_score(hypotheses, [references]).score,
        "chrfpp": chrf_metric.corpus_score(hypotheses, [references]).score,
        "ter": ter_metric.corpus_score(hypotheses, [references]).score,
    }

def aggregate_metrics(group: pd.DataFrame) -> dict[str, Any]:
    metrics = corpus_mt_metrics(group)
    metrics.update({
        "n": len(group),
        "exact_match_rate": group["exact_match"].mean(),
        "normalized_exact_match_rate": group["normalized_exact_match"].mean(),
        "empty_output_rate": group["empty_output"].mean(),
        "source_copy_rate": group["source_copy"].mean(),
        "prompt_leakage_rate": group["prompt_leakage"].mean(),
        "extra_explanation_rate": group["extra_explanation"].mean(),
        "avg_latency_seconds": group["latency_seconds"].mean(),
    })
    if "bertscore_f1" in group.columns:
        metrics["bertscore_f1"] = group["bertscore_f1"].mean()
    return metrics

In [ ]:
# BERTScore
# Uses a multilingual encoder so English and Turkish outputs can be scored in one pass.

from bert_score import score as bert_score

needs_bertscore = (
    "bertscore_f1" not in scored_predictions_df.columns
    or scored_predictions_df["bertscore_f1"].isna().any()
)

if needs_bertscore:
    scored_predictions_df["bertscore_precision"] = np.nan
    scored_predictions_df["bertscore_recall"] = np.nan
    scored_predictions_df["bertscore_f1"] = np.nan

    for model_key, group in tqdm(scored_predictions_df.groupby("model_key"), desc="BERTScore by model"):
        P, R, F1 = bert_score(
            group["prediction_text"].fillna("").astype(str).tolist(),
            group["reference_text"].fillna("").astype(str).tolist(),
            model_type="bert-base-multilingual-cased",
            batch_size=BERTSCORE_BATCH_SIZE,
            verbose=True,
            rescale_with_baseline=False,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
        idx = group.index
        scored_predictions_df.loc[idx, "bertscore_precision"] = P.cpu().numpy()
        scored_predictions_df.loc[idx, "bertscore_recall"] = R.cpu().numpy()
        scored_predictions_df.loc[idx, "bertscore_f1"] = F1.cpu().numpy()

scored_predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
print("Saved predictions with behavior metrics and BERTScore:", PREDICTIONS_PATH)
display(scored_predictions_df[["model_name", "direction", "bertscore_f1", "prediction_text", "reference_text"]].head())

In [ ]:
# Aggregate metrics

summary_rows = []
for (model_key, model_name), group in scored_predictions_df.groupby(["model_key", "model_name"], sort=False):
    row = {"model_key": model_key, "model_name": model_name, "group_type": "overall", "group_value": "overall"}
    row.update(aggregate_metrics(group))
    summary_rows.append(row)

metrics_summary_df = pd.DataFrame(summary_rows).sort_values("chrfpp", ascending=False)
metrics_summary_df.to_csv(METRICS_SUMMARY_PATH, index=False, encoding="utf-8-sig")
display(metrics_summary_df)

grouped_rows = []
for group_col in ["direction", "domain", "difficulty", "tone"]:
    for (model_key, model_name, group_value), group in scored_predictions_df.groupby(["model_key", "model_name", group_col], sort=False):
        row = {"model_key": model_key, "model_name": model_name, "group_type": group_col, "group_value": group_value}
        row.update(aggregate_metrics(group))
        grouped_rows.append(row)

grouped_metrics_df = pd.DataFrame(grouped_rows).sort_values(["group_type", "group_value", "chrfpp"], ascending=[True, True, False])
grouped_metrics_df.to_csv(GROUPED_METRICS_PATH, index=False, encoding="utf-8-sig")
display(grouped_metrics_df.head(30))

print("Saved", METRICS_SUMMARY_PATH)
print("Saved", GROUPED_METRICS_PATH)

In [ ]:
# External published benchmark table
# These rows are context only. They were not computed on our airplane-domain test set.

RETRIEVAL_DATE = "2026-06-05"
external_benchmark_rows = [
    # OPUS-MT English -> Turkish model card benchmarks
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "tatoeba-test-v2021-08-07", "metric": "BLEU", "value": 42.3, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "flores101-devtest", "metric": "BLEU", "value": 31.4, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "newsdev2016", "metric": "BLEU", "value": 21.9, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "newstest2016", "metric": "BLEU", "value": 23.4, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "newstest2017", "metric": "BLEU", "value": 25.4, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "newstest2018", "metric": "BLEU", "value": 22.6, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-en-tr", "direction": "English->Turkish", "dataset": "flores101-devtest", "metric": "chr-F", "value": 0.62829, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr", "retrieval_date": RETRIEVAL_DATE, "notes": "Published chr-F uses model-card scale, not this notebook's chrF++ setup."},

    # OPUS-MT Turkish -> English model card benchmarks
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "tatoeba-test-v2021-08-07", "metric": "BLEU", "value": 57.6, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "flores101-devtest", "metric": "BLEU", "value": 37.6, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "newsdev2016", "metric": "BLEU", "value": 32.1, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "newstest2016", "metric": "BLEU", "value": 29.3, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "newstest2017", "metric": "BLEU", "value": 29.7, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "newstest2018", "metric": "BLEU", "value": 30.7, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "retrieval_date": RETRIEVAL_DATE, "notes": "Published model-card score; external dataset."},
    {"model": "Helsinki-NLP/opus-mt-tc-big-tr-en", "direction": "Turkish->English", "dataset": "flores101-devtest", "metric": "chr-F", "value": 0.64152, "source_url": "https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en", "retrieval_date": RETRIEVAL_DATE, "notes": "Published chr-F uses model-card scale, not this notebook's chrF++ setup."},

    # Historical WMT17 Turkish-English system description examples
    {"model": "JAIST WMT17 combined phrase-based system", "direction": "Turkish->English", "dataset": "newstest2017", "metric": "BLEU", "value": 13.1, "source_url": "https://www.statmt.org/wmt17/pdf/WMT41.pdf", "retrieval_date": RETRIEVAL_DATE, "notes": "Historical WMT17 news-domain result; not directly comparable."},
    {"model": "JAIST WMT17 combined phrase-based system", "direction": "English->Turkish", "dataset": "newstest2017", "metric": "BLEU", "value": 10.4, "source_url": "https://www.statmt.org/wmt17/pdf/WMT41.pdf", "retrieval_date": RETRIEVAL_DATE, "notes": "Historical WMT17 news-domain result; not directly comparable."},
]

internet_benchmarks_df = pd.DataFrame(external_benchmark_rows)
required_cols = {"model", "direction", "dataset", "metric", "value", "source_url", "retrieval_date"}
missing = required_cols - set(internet_benchmarks_df.columns)
if missing:
    raise AssertionError(f"External benchmark table missing columns: {missing}")

internet_benchmarks_df.to_csv(INTERNET_BENCHMARKS_PATH, index=False, encoding="utf-8-sig")
display(internet_benchmarks_df)
print("Saved", INTERNET_BENCHMARKS_PATH)

In [ ]:
# Plots

plot_summary = metrics_summary_df.copy()
plot_summary["model_name"] = pd.Categorical(
    plot_summary["model_name"],
    categories=plot_summary.sort_values("chrfpp", ascending=False)["model_name"],
    ordered=True,
)

quality_long = plot_summary.melt(
    id_vars=["model_name"],
    value_vars=["bleu", "chrfpp", "ter"],
    var_name="metric",
    value_name="score",
)
plt.figure(figsize=(11, 5))
sns.barplot(data=quality_long, x="model_name", y="score", hue="metric")
plt.xticks(rotation=20, ha="right")
plt.title("Direct in-domain lexical MT metrics by model")
plt.ylabel("Score (TER is lower better; BLEU/chrF++ higher better)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lexical_metrics_by_model.png", dpi=180)
plt.show()

plt.figure(figsize=(9, 4))
sns.barplot(data=plot_summary, x="model_name", y="bertscore_f1")
plt.xticks(rotation=20, ha="right")
plt.title("BERTScore F1 by model")
plt.ylabel("BERTScore F1")
plt.ylim(max(0, plot_summary["bertscore_f1"].min() - 0.05), min(1.0, plot_summary["bertscore_f1"].max() + 0.03))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bertscore_by_model.png", dpi=180)
plt.show()

direction_plot = grouped_metrics_df[grouped_metrics_df["group_type"] == "direction"].copy()
plt.figure(figsize=(10, 5))
sns.barplot(data=direction_plot, x="group_value", y="chrfpp", hue="model_name")
plt.title("chrF++ by translation direction on airplane-domain held-out sample")
plt.xlabel("Direction")
plt.ylabel("chrF++")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "direction_chrfpp_by_model.png", dpi=180)
plt.show()

plt.figure(figsize=(9, 4))
sns.barplot(data=plot_summary, x="model_name", y="avg_latency_seconds")
plt.xticks(rotation=20, ha="right")
plt.title("Average latency per example")
plt.ylabel("Seconds")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latency_by_model.png", dpi=180)
plt.show()

print("Saved plots to", OUTPUT_DIR)

In [ ]:
# Build Markdown report

def df_to_markdown(df: pd.DataFrame, max_rows: int = 20) -> str:
    if len(df) > max_rows:
        df = df.head(max_rows)
    return df.to_markdown(index=False)

main_cols = [
    "model_name", "n", "bleu", "chrfpp", "ter", "bertscore_f1",
    "normalized_exact_match_rate", "empty_output_rate", "source_copy_rate",
    "prompt_leakage_rate", "extra_explanation_rate", "avg_latency_seconds",
]
summary_for_report = metrics_summary_df[main_cols].copy()
for col in summary_for_report.select_dtypes(include=[float]).columns:
    summary_for_report[col] = summary_for_report[col].round(4)

direction_for_report = grouped_metrics_df[grouped_metrics_df["group_type"] == "direction"][
    ["model_name", "group_value", "n", "bleu", "chrfpp", "ter", "bertscore_f1"]
].copy()
for col in direction_for_report.select_dtypes(include=[float]).columns:
    direction_for_report[col] = direction_for_report[col].round(4)

external_for_report = internet_benchmarks_df.copy()

report = f"""# Airplane Translation Evaluation Report

Generated on: {date.today().isoformat()}

## Evaluation Setup

The project is evaluated as a Turkish-English / English-Turkish airplane-domain translation system. The held-out split was recreated from `fine_tune_chat.jsonl` using the original training split settings: `test_size=0.02` and `seed=42`.

- Full held-out size: {len(heldout_df)} examples
- Evaluated examples in this run: {len(eval_df)} examples
- RUN_FULL_TEST_SET: {RUN_FULL_TEST_SET}
- Models evaluated directly: {', '.join(metrics_summary_df['model_name'].tolist())}

The direct comparison below is the main result because every model was evaluated on the same project-specific airplane test examples.

## Metric Definitions

- **BLEU**: n-gram overlap with the reference translation. Higher is better.
- **chrF++**: character n-gram and word n-gram F-score from SacreBLEU. Higher is better and often works well for morphologically rich languages such as Turkish.
- **TER**: translation edit rate. Lower is better.
- **BERTScore F1**: semantic similarity based on multilingual BERT embeddings. Higher is better.
- **Normalized exact match**: exact match after lowercasing, punctuation removal, and whitespace normalization. Higher is better.
- **Source-copy rate**: cases where the output is essentially the source sentence. Lower is better.
- **Prompt-leakage / extra-explanation rates**: app behavior checks for outputs that include prompt text, role text, explanations, or extra lines. Lower is better.

## Direct In-Domain Model Comparison

{df_to_markdown(summary_for_report)}

## Direction Breakdown

{df_to_markdown(direction_for_report, max_rows=50)}

## External Published Benchmarks

These internet benchmark results are included for context only. They are not direct evidence that one model is better or worse on this airplane-domain project because they use different datasets and sometimes different metric settings.

{df_to_markdown(external_for_report, max_rows=60)}

## Interpretation Guidance

For the final project report and presentation, use the direct in-domain table as the primary result. The external benchmark table can support background discussion by showing that OPUS-MT is a strong general Turkish-English translation baseline on public datasets, but it should not be mixed into the same ranking as the airplane-domain test results.

## Limitations

- The held-out split was already used as the training notebook's evaluation split, so it is validation-style rather than a fully blind test set.
- The airplane dataset was generated synthetically, so automatic metrics should be paired with a small human review of representative translations if possible.
- BLEU and exact match can penalize valid paraphrases. chrF++ and BERTScore help reduce that issue but are still automatic proxies.
- Internet benchmark values are from model cards or papers and may use different tokenization, datasets, and evaluation conventions.

## Output Files

- `predictions.csv`
- `metrics_summary.csv`
- `grouped_metrics.csv`
- `internet_benchmarks.csv`
- `lexical_metrics_by_model.png`
- `bertscore_by_model.png`
- `direction_chrfpp_by_model.png`
- `latency_by_model.png`
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)
print(report[:2500])

## Source Notes

Public benchmark rows were manually entered from these sources accessed on 2026-06-05:

- OPUS-MT English->Turkish model card: https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr
- OPUS-MT Turkish->English model card: https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-tr-en
- JAIST WMT17 Turkish-English system paper: https://www.statmt.org/wmt17/pdf/WMT41.pdf
- SacreBLEU implementation: https://github.com/mjpost/sacrebleu
- BERTScore paper: https://arxiv.org/abs/1904.09675

If you add new public benchmark rows, keep them in `internet_benchmarks.csv` and include the source URL, retrieval date, dataset, metric, and notes.